# Stage 2: Cleaning, Splitting, Preprocessing, Augmentation and Data Loaders
**Deep Learning-Based Detection and Classification of Brain Tumors Using MRI Images**

Based on the Stage 1 inspection (byte-identical duplicates, a leaking Train/Val split, no test set, no patient IDs).
This notebook (1) removes exact duplicates, (2) groups near-duplicates, (3) builds a group-aware stratified 70/15/15 split,
(4) verifies there is no leakage, (5) runs a shortcut diagnostic, (6) builds preprocessing/augmentation and data loaders,
(7) verifies the loaders before any training.

Runtime: GPU. The source files in `src/` are written by the first cells, so nothing needs to be uploaded.

## 1. Create project folders and write the source modules

In [ ]:
from pathlib import Path
PROJECT = Path('/content/brain-tumor-project')
for d in ['data','notebooks','src','models','results/figures','results/metrics','results/predictions']:
    (PROJECT/d).mkdir(parents=True, exist_ok=True)

In [ ]:
%%writefile /content/brain-tumor-project/src/utils.py
"""Reproducibility helpers."""
import os, random, platform
import numpy as np


def set_seed(seed=42):
    """Seed Python, NumPy, PyTorch and CUDA. Deterministic cuDNN is requested (can be slightly slower)."""
    import torch
    random.seed(seed); np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def env_info(seed=42):
    import torch, torchvision
    return dict(python=platform.python_version(), torch=torch.__version__, torchvision=torchvision.__version__,
                cuda_available=torch.cuda.is_available(),
                gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, seed=seed)

In [ ]:
%%writefile /content/brain-tumor-project/src/data_prep.py
"""Dataset inventory, duplicate handling, grouping and splitting.

This module does not need PyTorch. All numbers it reports are computed from
the files on disk. The folder layout it expects was verified in Stage 1:
    <root>/{Train,Val}/<ClassName>/{images,labels}/<file>
and it raises a clear error if the layout is different.
"""
import hashlib, json
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image

CLASS_NAMES = ['Glioma', 'Meningioma', 'No Tumor', 'Pituitary']   # alphabetical, same order as the box class ids
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}


def find_dataset_root(base):
    """Return the folder that directly contains the Train and Val folders."""
    base = Path(base)
    for p in [base] + [q for q in base.rglob('*') if q.is_dir()]:
        names = {c.name.lower() for c in p.iterdir() if c.is_dir()}
        if {'train', 'val'} <= names:
            return p
    raise FileNotFoundError(f'No folder containing Train/ and Val/ found under {base}')


def _dhash(img):
    a = np.asarray(img.convert('L').resize((9, 8), Image.LANCZOS), dtype=np.int16)
    bits = (a[:, 1:] > a[:, :-1]).flatten()
    return int(np.packbits(bits).view('>u8')[0])


def _md5(path):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()


def _read_boxes(txt_path):
    """YOLO lines: class cx cy w h (normalised). Returns a list of lists."""
    if txt_path is None:
        return []
    boxes = []
    for line in txt_path.read_text().strip().splitlines():
        t = line.split()
        if len(t) >= 5:
            boxes.append([int(float(t[0]))] + [float(v) for v in t[1:5]])
    return boxes


def build_inventory(root, progress=lambda x, **k: x):
    """One row per image: path, class, original split, size, hashes, boxes."""
    root = Path(root)
    label_files = {p.stem: p for p in root.rglob('*.txt')}
    paths = sorted(p for p in root.rglob('*') if p.is_file() and p.suffix.lower() in IMG_EXT)
    rows, unreadable = [], []
    for p in progress(paths, desc='inventory'):
        rel = p.relative_to(root)
        if len(rel.parts) < 3 or rel.parts[1] not in CLASS_TO_IDX:
            raise ValueError(f'Unexpected path layout: {rel}. Expected <Split>/<Class>/images/<file>.')
        try:
            with Image.open(p) as im:
                im.verify()
            with Image.open(p) as im:
                w, h = im.size
                mode, fmt = im.mode, im.format
                dh = _dhash(im)
                rgb = np.asarray(im.convert('RGB'))
        except Exception as e:
            unreadable.append((str(rel), repr(e)))
            continue
        boxes = _read_boxes(label_files.get(p.stem))
        rows.append(dict(
            rel=str(rel), split_orig=rel.parts[0].lower(), label=rel.parts[1],
            label_idx=CLASS_TO_IDX[rel.parts[1]], stem=p.stem, width=w, height=h, mode=mode, format=fmt,
            gray_like=bool(np.array_equal(rgb[..., 0], rgb[..., 1]) and np.array_equal(rgb[..., 1], rgb[..., 2])),
            mean=float(rgb.mean()), std=float(rgb.std()), size_kb=p.stat().st_size / 1024,
            md5=_md5(p), dhash=dh, has_label_file=p.stem in label_files,
            n_boxes=len(boxes), boxes_json=json.dumps(boxes)))
    df = pd.DataFrame(rows)
    df['dhash'] = df['dhash'].astype(np.uint64)
    return df, unreadable


def drop_exact_duplicates(df):
    """Keep one file per identical-bytes group (files from Train/ are preferred, then by path).
    Groups whose copies carry different class labels are removed entirely (none expected)."""
    df = df.copy()
    df['_rank'] = (df.split_orig != 'train').astype(int)
    df = df.sort_values(['_rank', 'rel']).reset_index(drop=True)
    n_labels = df.groupby('md5').label.nunique()
    conflict_md5 = set(n_labels[n_labels > 1].index)
    df = df[~df.md5.isin(conflict_md5)]
    keep = ~df.md5.duplicated(keep='first')
    kept = df[keep][['md5', 'rel']].rename(columns={'rel': 'kept_rel'})
    log = df[~keep][['rel', 'label', 'split_orig', 'md5']].merge(kept, on='md5')
    out = df[keep].drop(columns='_rank').reset_index(drop=True)
    return out, log, len(conflict_md5)


def near_duplicate_groups(root, df, dhash_thr=2, corr_thr=0.95, thumb=64):
    """Group images that are visually near-identical although their bytes differ.
    A pair is linked only if (a) dHash Hamming distance <= dhash_thr AND
    (b) Pearson correlation of 64x64 grayscale thumbnails >= corr_thr.
    Linked pairs are merged into connected components (union-find)."""
    root = Path(root)
    H = df.dhash.values.astype(np.uint64)
    n = len(H)
    pop = np.array([bin(i).count('1') for i in range(256)], dtype=np.uint8)
    cand = []
    for i in range(0, n, 500):
        x = H[i:i + 500, None] ^ H[None, :]
        d = pop[x.view(np.uint8).reshape(x.shape[0], x.shape[1], 8)].sum(axis=2)
        ii, jj = np.where(d <= dhash_thr)
        cand += [(a + i, b, int(d[a, b])) for a, b in zip(ii, jj) if a + i < b]
    cache = {}
    def thumb_of(k):
        if k not in cache:
            im = Image.open(root / df.rel.iloc[k]).convert('L').resize((thumb, thumb))
            cache[k] = np.asarray(im, dtype=np.float32).ravel()
        return cache[k]
    parent = list(range(n))
    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]; a = parent[a]
        return a
    pairs = []
    for a, b, d in cand:
        x, y = thumb_of(a), thumb_of(b)
        c = float(np.corrcoef(x, y)[0, 1]) if x.std() > 0 and y.std() > 0 else float('nan')
        linked = c >= corr_thr
        pairs.append(dict(i=a, j=b, dhash_dist=d, corr=c, linked=linked,
                          label_i=df.label.iloc[a], label_j=df.label.iloc[b]))
        if linked:
            parent[find(a)] = find(b)
    group = np.array([find(k) for k in range(n)])
    _, group = np.unique(group, return_inverse=True)
    return group, pd.DataFrame(pairs)


def make_splits(df, seed=42, n_splits=7):
    """Stratified, group-aware split. With 7 folds: 1 fold test (~14.3%), 1 fold val (~14.3%),
    5 folds train (~71.4%), i.e. approximately 70/15/15. Images in the same group never cross splits."""
    from sklearn.model_selection import StratifiedGroupKFold
    df = df.copy()
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    fold = np.full(len(df), -1)
    for k, (_, te) in enumerate(sgkf.split(df, df.label_idx, df.group)):
        fold[te] = k
    df['split'] = np.where(fold == 0, 'test', np.where(fold == 1, 'val', 'train'))
    return df


def verify_splits(df, pairs=None):
    """Leakage checks on the final split table. Returns a dict of results."""
    res = {}
    res['n_images'] = len(df)
    res['counts'] = df.split.value_counts().to_dict()
    res['class_counts'] = pd.crosstab(df.split, df.label).to_dict('index')
    res['md5_shared_across_splits'] = int((df.groupby('md5').split.nunique() > 1).sum())
    res['groups_shared_across_splits'] = int((df.groupby('group').split.nunique() > 1).sum())
    res['rel_duplicated'] = int(df.rel.duplicated().sum())
    if pairs is not None and len(pairs):
        lk = pairs[pairs.linked]
        sp = df.split.values
        res['linked_near_dup_pairs'] = int(len(lk))
        res['linked_near_dup_pairs_across_splits'] = int((sp[lk.i.values] != sp[lk.j.values]).sum())
        res['linked_pairs_with_different_labels'] = int((lk.label_i != lk.label_j).sum())
    return res

In [ ]:
%%writefile /content/brain-tumor-project/src/shortcut_check.py
"""Shortcut diagnostic: how well can the class be predicted WITHOUT looking at anatomy?
Uses only image metadata and global pixel statistics. Fit on the train split, evaluated on the
validation split (the test split is not touched). A high score means a CNN could also exploit
such non-anatomical cues, so its accuracy must be interpreted with care."""
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

FEATURE_SETS = {
    'majority class (reference)': [],
    'image size only (width, height, aspect, is_512x512)': ['width', 'height', 'aspect', 'is_512'],
    'size + colour mode + file size': ['width', 'height', 'aspect', 'is_512', 'mode_code', 'gray_like', 'size_kb'],
    'size + mode + file size + global intensity (mean, std)': ['width', 'height', 'aspect', 'is_512', 'mode_code', 'gray_like', 'size_kb', 'mean', 'std'],
}

def run(df, seed=42):
    d = df.copy()
    d['aspect'] = d.width / d.height
    d['is_512'] = ((d.width == 512) & (d.height == 512)).astype(int)
    d['mode_code'] = d['mode'].astype('category').cat.codes
    d['gray_like'] = d.gray_like.astype(int)
    tr, va = d[d.split == 'train'], d[d.split == 'val']
    rows = []
    for name, feats in FEATURE_SETS.items():
        if feats:
            clf = RandomForestClassifier(n_estimators=300, random_state=seed, n_jobs=-1, class_weight='balanced')
            clf.fit(tr[feats], tr.label_idx); pred = clf.predict(va[feats])
        else:
            clf = DummyClassifier(strategy='most_frequent').fit(tr[['width']], tr.label_idx); pred = clf.predict(va[['width']])
        rows.append(dict(features=name, val_accuracy=accuracy_score(va.label_idx, pred),
                         val_macro_f1=f1_score(va.label_idx, pred, average='macro')))
        last_pred = pred
    return pd.DataFrame(rows), last_pred, va

In [ ]:
%%writefile /content/brain-tumor-project/src/preprocessing.py
"""Image preprocessing and augmentation.

Training preprocessing : pad to square -> resize -> mild random augmentation -> tensor -> normalise
Validation/test        : pad to square -> resize -> tensor -> normalise      (NO random operations)

Why pad instead of stretch: the images are not all square (for example 201x251), and stretching would
distort the shape of the head and of the tumour. Padding with black keeps the aspect ratio.
Why 3 channels: ImageNet-pretrained ResNets expect RGB input, so grayscale MRI is replicated to 3 channels
(done when the file is loaded, with PIL's convert('RGB')).
Why ImageNet mean/std: the pretrained weights were trained with this normalisation.
"""
from PIL import Image

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


class PadToSquare:
    """Pad a PIL image with a constant colour (default black) to a centred square."""
    def __init__(self, fill=0):
        self.fill = fill

    def __call__(self, img):
        w, h = img.size
        s = max(w, h)
        if w == h:
            return img
        canvas = Image.new(img.mode, (s, s), (self.fill,) * len(img.getbands()))
        canvas.paste(img, ((s - w) // 2, (s - h) // 2))
        return canvas


def get_transforms(train, img_size=224, hflip=False):
    """Return a torchvision transform. hflip is OFF by default (anatomical orientation, see report)."""
    from torchvision import transforms as T
    base = [PadToSquare(0), T.Resize((img_size, img_size))]
    tail = [T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)]
    if not train:
        return T.Compose(base + tail)
    aug = [
        T.RandomResizedCrop(img_size, scale=(0.85, 1.0), ratio=(0.95, 1.05)),   # moderate crop
        T.RandomAffine(degrees=10, translate=(0.05, 0.05)),                      # small rotation / shift
        T.ColorJitter(brightness=0.1, contrast=0.1),                             # slight intensity change
    ]
    if hflip:
        aug.insert(0, T.RandomHorizontalFlip(0.5))   # only for the separate ablation experiment
    return T.Compose(base + aug + tail)


def denormalize(t):
    """Tensor (3,H,W) normalised -> numpy (H,W,3) in [0,1] for plotting."""
    import torch
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1); std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (t * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

In [ ]:
%%writefile /content/brain-tumor-project/src/dataset.py
"""PyTorch Dataset and DataLoader construction."""
from pathlib import Path
import json
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from preprocessing import get_transforms


class MRIDataset(Dataset):
    """Reads rows of the split table. Images are opened lazily and converted to RGB."""
    def __init__(self, df, root, transform=None, return_meta=False):
        self.df = df.reset_index(drop=True)
        self.root = Path(root)
        self.transform = transform
        self.return_meta = return_meta

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(self.root / row.rel).convert('RGB')   # L / RGBA / P -> RGB
        if self.transform is not None:
            img = self.transform(img)
        label = int(row.label_idx)
        if self.return_meta:
            return img, label, dict(rel=row.rel, boxes=json.loads(row.boxes_json))
        return img, label


def _worker_init(worker_id):
    seed = torch.initial_seed() % 2**32
    np.random.seed(seed); import random; random.seed(seed)


def class_weights(train_df, n_classes=4):
    """Inverse-frequency weights (only used if the imbalance experiment is run)."""
    counts = np.bincount(train_df.label_idx.values, minlength=n_classes).astype(float)
    return torch.tensor(counts.sum() / (n_classes * counts), dtype=torch.float32)


def make_loaders(split_df, root, batch_size=32, img_size=224, hflip=False, num_workers=2, seed=42, sampler=None):
    """Return dict of train/val/test loaders. Only the training set uses random augmentation."""
    g = torch.Generator(); g.manual_seed(seed)
    loaders = {}
    for name in ['train', 'val', 'test']:
        part = split_df[split_df.split == name]
        ds = MRIDataset(part, root, get_transforms(train=(name == 'train'), img_size=img_size, hflip=hflip))
        loaders[name] = DataLoader(
            ds, batch_size=batch_size,
            shuffle=(name == 'train' and sampler is None),
            sampler=sampler if name == 'train' else None,
            num_workers=num_workers, pin_memory=torch.cuda.is_available(),
            worker_init_fn=_worker_init, generator=g if name == 'train' else None,
            drop_last=False)
    return loaders

## 2. Imports, seed and environment record

In [ ]:
import sys, json, warnings
sys.path.insert(0, str(PROJECT/'src'))
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, torch
from collections import Counter
from tqdm.auto import tqdm
import data_prep as dp, shortcut_check as sc
from utils import set_seed, env_info
from preprocessing import get_transforms, PadToSquare, denormalize, IMAGENET_MEAN, IMAGENET_STD
from dataset import MRIDataset, make_loaders, class_weights

SEED = 42
set_seed(SEED)
ENV = env_info(SEED); print(json.dumps(ENV, indent=2))

## 3. Dataset setup
Set `DATA_SOURCE` to `'kaggle'` (Colab Secrets `KAGGLE_USERNAME` and `KAGGLE_KEY`, or upload `kaggle.json`) or `'drive'` (a zip in Google Drive).
No credentials are stored in the notebook.

In [ ]:
import os, subprocess, zipfile
DATA_SOURCE = 'kaggle'                                   # 'kaggle' or 'drive'
DRIVE_ZIP = '/content/drive/MyDrive/brain_tumor_dataset.zip'   # only used if DATA_SOURCE == 'drive'
SLUG = 'ahmedsorour1/mri-for-brain-tumor-with-bounding-boxes'
RAW = PROJECT/'data'/'raw'; RAW.mkdir(exist_ok=True)

if not any(p.is_file() for p in RAW.rglob('*')):
    if DATA_SOURCE == 'drive':
        from google.colab import drive; drive.mount('/content/drive')
        assert Path(DRIVE_ZIP).exists(), f'Not found: {DRIVE_ZIP}'
        with zipfile.ZipFile(DRIVE_ZIP) as f: f.extractall(RAW)
    else:
        if not (os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY')):
            try:
                from google.colab import userdata
                os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME'); os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
            except Exception:
                from google.colab import files
                print('Upload kaggle.json'); up = files.upload()
                kd = Path.home()/'.kaggle'; kd.mkdir(exist_ok=True); (kd/'kaggle.json').write_bytes(up[list(up)[0]]); os.chmod(kd/'kaggle.json', 0o600)
        subprocess.run(['pip','-q','install','kaggle'], check=True)
        subprocess.run(['kaggle','datasets','download','-d',SLUG,'-p',str(RAW),'--unzip'], check=True)
else:
    print('Data already present, skipping download.')

ROOT = dp.find_dataset_root(RAW)
print('Dataset root:', ROOT, '| subfolders:', sorted(c.name for c in ROOT.iterdir() if c.is_dir()))

## 4. Inventory (reads every image once: size, hashes, annotation file)

In [ ]:
inv, unreadable = dp.build_inventory(ROOT, progress=tqdm)
print('Images:', len(inv), '| unreadable:', len(unreadable))
print('Images without a label file:', int((~inv.has_label_file).sum()), '| images without boxes:', int((inv.n_boxes == 0).sum()))
print(inv.groupby(['split_orig','label']).size().unstack())

## 5. Remove exact duplicates
Byte-identical files are redundant and, when they fall on both sides of a split, inflate the evaluation score. One copy is kept
(preferring the one in `Train/`). The removal log is saved.

In [ ]:
clean, dup_log, n_conflict = dp.drop_exact_duplicates(inv)
print(f'Before: {len(inv)} | after: {len(clean)} | removed: {len(dup_log)} | duplicate groups with conflicting labels removed: {n_conflict}')
print('Images per class after de-duplication:', clean.label.value_counts().to_dict())
dup_log.to_csv(PROJECT/'results/metrics/exact_duplicates_removed.csv', index=False)

## 6. Group near-duplicates
Images with a very similar perceptual hash (dHash distance <= 2) are linked only when a pixel-level check also agrees
(correlation of 64x64 grayscale thumbnails >= 0.95). Linked images form a group, and a group is never split across train/val/test.
No patient IDs exist, so these groups are the closest available substitute for patient-level grouping. They cannot catch
every pair of slices from the same patient.

In [ ]:
group, pairs = dp.near_duplicate_groups(ROOT, clean, dhash_thr=2, corr_thr=0.95)
clean['group'] = group
linked = pairs[pairs.linked]
gsize = clean.groupby('group').size()
print('Candidate pairs (dHash<=2):', len(pairs), '| confirmed by pixel check:', len(linked),
      '| confirmed pairs with different labels:', int((linked.label_i != linked.label_j).sum()))
print('Groups:', len(gsize), '| groups with more than one image:', int((gsize > 1).sum()), '| largest group:', int(gsize.max()))
print('Groups with mixed labels:', int((clean.groupby('group').label.nunique() > 1).sum()))
pairs.to_csv(PROJECT/'results/metrics/near_duplicate_pairs.csv', index=False)

## 7. Stratified, group-aware 70/15/15 split (seed 42)
`StratifiedGroupKFold` with 7 folds: 1 fold for test, 1 fold for validation, 5 folds for training. This gives about 71.4% / 14.3% / 14.3%,
keeps class proportions, and keeps each group inside one split. The original `Train/Val` folders are not used because the validation
folder contained copies of training images (Stage 1).

In [ ]:
splits = dp.make_splits(clean, seed=SEED, n_splits=7)
check = dp.verify_splits(splits, pairs)
print(json.dumps(check, indent=1, default=str))
assert check['md5_shared_across_splits'] == 0
assert check['groups_shared_across_splits'] == 0
assert check['rel_duplicated'] == 0
assert check.get('linked_near_dup_pairs_across_splits', 0) == 0
ct = pd.crosstab(splits.split, splits.label, margins=True); print(ct)
print((pd.crosstab(splits.split, splits.label, normalize='index')*100).round(1))
splits.to_csv(PROJECT/'results/metrics/splits.csv', index=False)

In [ ]:
order = ['train','val','test']
plot_df = pd.crosstab(splits.split, splits.label).loc[order]
ax = plot_df.plot(kind='bar', figsize=(7,4), width=.8); plt.xticks(rotation=0)
for c in ax.containers: ax.bar_label(c, fontsize=8)
plt.title('Images per class in each split (after de-duplication)'); plt.ylabel('Images'); plt.xlabel('Split'); plt.legend(title='Class')
plt.tight_layout(); plt.savefig(PROJECT/'results/figures/split_class_distribution.png', dpi=200); plt.show()

## 8. Shortcut diagnostic
A Random Forest sees **only** image size, colour mode, file size and global intensity (no anatomy). It is fitted on the training split and scored on
the validation split. If it scores well above the majority-class reference, a CNN can also exploit non-anatomical cues, and the CNN results
must be interpreted with that in mind.

In [ ]:
short, short_pred, short_va = sc.run(splits, seed=SEED)
print(short.round(3).to_string(index=False))
from sklearn.metrics import confusion_matrix
print(dp.CLASS_NAMES); print(confusion_matrix(short_va.label_idx, short_pred))
short.to_csv(PROJECT/'results/metrics/shortcut_diagnostic.csv', index=False)

## 9. Preprocessing and augmentation check
Left: original. Next: validation/test version (pad to square, resize to 224, no randomness). Right: three random training augmentations.

In [ ]:
from PIL import Image
set_seed(SEED)
tf_eval, tf_train = get_transforms(False), get_transforms(True)
fig, axes = plt.subplots(4, 5, figsize=(13, 11))
for r, cls in enumerate(dp.CLASS_NAMES):
    row = splits[(splits.split=='train') & (splits.label==cls)].sample(1, random_state=SEED).iloc[0]
    img = Image.open(ROOT/row.rel).convert('RGB')
    panels = [('original %dx%d' % img.size, np.asarray(img)), ('eval (pad+resize)', denormalize(tf_eval(img)))] + \
             [(f'train aug {k+1}', denormalize(tf_train(img))) for k in range(3)]
    for c, (t, a) in enumerate(panels):
        axes[r][c].imshow(a); axes[r][c].axis('off'); axes[r][c].set_title(f'{cls}: {t}' if c == 0 else t, fontsize=8)
plt.tight_layout(); plt.savefig(PROJECT/'results/figures/preprocessing_augmentation_examples.png', dpi=200); plt.show()

## 10. Data loader verification (must pass before any training)

In [ ]:
BATCH_SIZE, IMG_SIZE = 32, 224
loaders = make_loaders(splits, ROOT, batch_size=BATCH_SIZE, img_size=IMG_SIZE, hflip=False, num_workers=2, seed=SEED)
for k, dl in loaders.items(): print(f'{k:5s}: {len(dl.dataset):5d} images, {len(dl):3d} batches')

xb, yb = next(iter(loaders['train']))
print('\nBatch tensor:', tuple(xb.shape), xb.dtype, '| labels:', tuple(yb.shape), yb.dtype)
print('Value range: %.2f to %.2f | per-channel mean %s' % (xb.min(), xb.max(), xb.mean(dim=(0,2,3)).numpy().round(2)))
assert xb.shape[1:] == (3, IMG_SIZE, IMG_SIZE) and torch.isfinite(xb).all()

# evaluation pipeline is deterministic, training pipeline is random
v = loaders['val'].dataset
assert torch.equal(v[0][0], v[0][0]), 'validation transform is not deterministic'
t = loaders['train'].dataset
assert not torch.equal(t[0][0], t[0][0]), 'training augmentation does not seem to be active'

# label counts seen through the loaders equal the split table; no file appears in two loaders
seen = {}
for k, dl in loaders.items():
    cnt = Counter()
    for _, y in tqdm(dl, desc=k, leave=False): cnt.update(y.tolist())
    expected = splits[splits.split==k].label_idx.value_counts().to_dict()
    assert dict(cnt) == expected, (k, dict(cnt), expected)
    seen[k] = set(dl.dataset.df.rel)
assert not (seen['train'] & seen['val']) and not (seen['train'] & seen['test']) and not (seen['val'] & seen['test'])
print('\nAll loader checks passed.')

tr_df = splits[splits.split=='train']
print('Class counts (train):', tr_df.label.value_counts().to_dict())
print('Imbalance ratio (largest/smallest, train): %.2f' % (tr_df.label.value_counts().max()/tr_df.label.value_counts().min()))
print('Inverse-frequency class weights (not used unless the imbalance experiment is run):', dict(zip(dp.CLASS_NAMES, class_weights(tr_df).numpy().round(3))))

## 11. Save the Stage 2 configuration

In [ ]:
config = dict(
    seed=SEED, environment=ENV, class_names=dp.CLASS_NAMES, img_size=IMG_SIZE, batch_size=BATCH_SIZE,
    normalisation=dict(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    eval_preprocessing='pad to square (black) -> resize 224x224 -> tensor -> normalise',
    train_augmentation='RandomResizedCrop(scale 0.85-1.0, ratio 0.95-1.05), RandomAffine(10 deg, translate 5%), ColorJitter(brightness 0.1, contrast 0.1); no horizontal flip',
    cleaning=dict(images_found=len(inv), exact_duplicates_removed=len(dup_log), images_after_cleaning=len(clean)),
    near_duplicate_grouping=dict(dhash_threshold=2, thumbnail_corr_threshold=0.95, confirmed_pairs=int(len(linked)), groups=int(len(gsize))),
    split=dict(method='StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=42): fold0=test, fold1=val, rest=train',
               counts=splits.split.value_counts().to_dict(), verification=check),
    shortcut_diagnostic=short.round(4).to_dict('records'))
json.dump(config, open(PROJECT/'results/metrics/stage2_config.json','w'), indent=2, default=str)
print('Saved:', PROJECT/'results/metrics/stage2_config.json')